In [1]:
import numpy as np
import pandas as pd

In [2]:
df= pd.read_csv('moviereviews.csv')
df.head()

,label,review
0,neg,how do films like mouse hunt get into theatres...
1,neg,some talented actresses are blessed with a dem...
2,pos,this has been an extraordinary year for austra...
3,pos,according to hollywood movies made in last few...
4,neg,my first press screening of 1998 and already i...


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   2000 non-null   object
 1   review  1965 non-null   object
dtypes: object(2)
memory usage: 31.4+ KB


In [4]:
df.isna().sum()

label      0
review    35
dtype: int64

In [5]:
df= df.dropna()

In [6]:
df[df['review'].str.isspace()]

,label,review
57,neg,
71,pos,
147,pos,
151,pos,
283,pos,
307,pos,
313,neg,
323,pos,
343,pos,
351,neg,


In [8]:
df[~df['review'].str.isspace()]

,label,review
0,neg,how do films like mouse hunt get into theatres...
1,neg,some talented actresses are blessed with a dem...
2,pos,this has been an extraordinary year for austra...
3,pos,according to hollywood movies made in last few...
4,neg,my first press screening of 1998 and already i...
...,...,...
1995,pos,"i like movies with albert brooks , and i reall..."
1996,pos,it might surprise some to know that joel and e...
1997,pos,the verdict : spine-chilling drama from horror...
1998,pos,i want to correct what i wrote in a former ret...


In [9]:
df= df[~df['review'].str.isspace()]

In [10]:
df.head()

,label,review
0,neg,how do films like mouse hunt get into theatres...
1,neg,some talented actresses are blessed with a dem...
2,pos,this has been an extraordinary year for austra...
3,pos,according to hollywood movies made in last few...
4,neg,my first press screening of 1998 and already i...


In [12]:
df['label'].value_counts()

label
neg    969
pos    969
Name: count, dtype: int64

### Bag of Words

In [13]:
from sklearn.feature_extraction.text import CountVectorizer

In [14]:
cv= CountVectorizer(stop_words='english')

In [15]:
df[df['label']=='neg']

,label,review
0,neg,how do films like mouse hunt get into theatres...
1,neg,some talented actresses are blessed with a dem...
4,neg,my first press screening of 1998 and already i...
5,neg,"to put it bluntly , ed wood would have been pr..."
6,neg,"synopsis : melissa , a mentally-disturbed woma..."
...,...,...
1985,neg,"the real blonde ( r ) a woman's face , an arm ..."
1986,neg,* * * the following review contains spoilers ...
1987,neg,""" book "" should have remained in shadows \r\n..."
1991,neg,"all right , all right , we get the point : des..."


In [16]:
df[df['label']=='neg']['review']

0       how do films like mouse hunt get into theatres...
1       some talented actresses are blessed with a dem...
4       my first press screening of 1998 and already i...
5       to put it bluntly , ed wood would have been pr...
6       synopsis : melissa , a mentally-disturbed woma...
                              ...                        
1985    the real blonde ( r ) a woman's face , an arm ...
1986     * * * the following review contains spoilers ...
1987     " book " should have remained in shadows \r\n...
1991    all right , all right , we get the point : des...
1992    say , tell me if you've seen this before : a c...
Name: review, Length: 969, dtype: object

In [17]:
cv.fit_transform(df[df['label']=='neg']['review'])

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 223948 stored elements and shape (969, 27473)>

In [24]:
matrix= cv.fit_transform(df[df['label']=='neg']['review'])

freqs= zip(cv.get_feature_names_out(),matrix.sum(axis=0).tolist()[0])

print(f"Top 20 words used for negative reviews: {sorted(freqs,key=lambda x:-x[1])[:30]}")

Top 20 words used for negative reviews: [('film', 4063), ('movie', 3131), ('like', 1808), ('just', 1480), ('time', 1127), ('good', 1117), ('bad', 997), ('character', 926), ('story', 908), ('plot', 888), ('characters', 838), ('make', 813), ('really', 743), ('way', 734), ('little', 696), ('don', 683), ('does', 666), ('doesn', 648), ('action', 635), ('scene', 634), ('people', 628), ('director', 627), ('films', 623), ('know', 617), ('scenes', 608), ('man', 607), ('big', 583), ('new', 553), ('movies', 544), ('better', 514)]


In [25]:
matrix= cv.fit_transform(df[df['label']=='pos']['review'])

freqs= zip(cv.get_feature_names_out(),matrix.sum(axis=0).tolist()[0])

print(f"Top 20 words used for positive reviews: {sorted(freqs,key=lambda x:-x[1])[:30]}")

Top 20 words used for positive reviews: [('film', 5002), ('movie', 2389), ('like', 1721), ('just', 1273), ('story', 1199), ('good', 1193), ('time', 1175), ('character', 1037), ('life', 1032), ('characters', 957), ('way', 864), ('films', 851), ('does', 828), ('best', 788), ('people', 769), ('make', 764), ('little', 751), ('really', 731), ('man', 728), ('new', 702), ('great', 692), ('scene', 675), ('world', 646), ('love', 634), ('movies', 604), ('scenes', 604), ('plot', 577), ('doesn', 576), ('director', 564), ('don', 555)]


### Training the model

In [26]:
from sklearn.model_selection import train_test_split

X=df['review']
y=df['label']

X_train,X_test,y_train,y_test= train_test_split(X,y,test_size=0.2,random_state=50)

In [27]:
from sklearn.pipeline import Pipeline

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB

In [28]:
pipe= Pipeline([('tfidf',TfidfVectorizer()),('nb',MultinomialNB())])

In [29]:
pipe.fit(X_train,y_train)

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('nb', MultinomialNB())])

In [30]:
from sklearn.metrics import classification_report,confusion_matrix

In [31]:
y_pred= pipe.predict(X_test)

In [32]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

         neg       0.91      0.77      0.83       205
         pos       0.78      0.91      0.84       183

    accuracy                           0.84       388
   macro avg       0.84      0.84      0.84       388
weighted avg       0.85      0.84      0.84       388



In [33]:
confusion_matrix(y_pred,y_test)

array([[158,  16],
       [ 47, 167]])

In [43]:
new_review=['excellent movie','great seats','horrible experience','great writing, nice direction but bad action']

In [44]:
pipe.predict(new_review)

array(['pos', 'pos', 'neg', 'neg'], dtype='<U3')